
# VinBigData Chest X‑ray Abnormality Classification

This notebook provides a complete PyTorch pipeline for **multi‑label classification** of chest X‑ray (CXR) images from the **VinBigData Chest X‑ray Abnormalities Detection** dataset.  
The original Kaggle competition required participants to detect and localise 14 thoracic abnormalities from 18,000 postero‑anterior (PA) chest X‑rays.  
For classification, the bounding box annotations can be aggregated at the image level to produce a binary vector of length 15 (14 abnormalities plus the "No finding" class) indicating which findings are present in a radiograph【921128788067047†L822-L829】.  

Key characteristics of the dataset include:

* **Size and split** – The competition provided 15,000 labelled images for training and 3,000 images for testing.  A typical split uses 70% of the training images for training, 10% for validation and 20% for testing【921128788067047†L822-L829】.  
* **Multi‑label targets** – Many CXRs contain multiple abnormalities (average label cardinality ≈ 1.73).  A 'No finding' label is present when no abnormalities are annotated【921128788067047†L822-L829】.  
* **Severe class imbalance** – Common findings like *aortic enlargement* and *cardiomegaly* occur in more than 15% of images, while rare conditions like *pneumothorax* are found in less than 1% of the training data【921128788067047†L790-L816】.  
* **Preprocessing** – Images are stored as DICOM files.  A typical preprocessing pipeline reads pixel data, applies the rescale slope and intercept, converts to 8‑bit grayscale, inverts MONOCHROME1 images, applies slight Gaussian blur and **contrast‑limited adaptive histogram equalisation (CLAHE)**, and resizes to a fixed resolution (e.g. 384×384)【921128788067047†L840-L853】.  
* **Augmentation** – Mild geometric and photometric augmentations (random resized crop, horizontal flip, small rotations, brightness/contrast jitter) are applied during training, and optional test‑time augmentation (TTA) such as horizontal flipping can stabilise predictions【921128788067047†L869-L883】.  

State‑of‑the‑art solutions for the VinBigData classification task include hybrid architectures that fuse convolutional backbones with vision transformers (e.g. ConvNeXtV2–ViT).  When trained with weighted binary cross‑entropy, mixup regularisation and a cosine learning rate schedule, such models achieved **macro‑AUROC of 0.9525** and **micro‑AUROC of 0.9777** on the held‑out VinBigData test set【921128788067047†L210-L228】.  

In this notebook we implement a strong yet computationally tractable baseline using a single convolutional backbone from the `timm` library (e.g. ConvNeXt, EfficientNet V2 or ResNet50).  The key components are:

* **Multi‑label stratified splitting** to create train/validation/test sets that preserve label frequencies.
* **Class‑weighted loss** – `BCEWithLogitsLoss` with a per‑label `pos_weight` derived from the training set to penalise false negatives in rare classes【921128788067047†L793-L816】.
* **Mixup augmentation** to improve generalisation and mitigate overfitting【921128788067047†L818-L820】.
* **Learning rate schedule** – AdamW optimiser with cosine annealing and warm‑up【921128788067047†L1013-L1021】.
* **Evaluation** – per‑class AUROC, macro‑AUROC and micro‑AUROC.

> **Note:** Running the entire training loop on the full VinBigData dataset requires a GPU with substantial memory and several hours of training.  This notebook provides all the necessary functions; you can adapt batch size and model architecture to suit your hardware.


In [18]:
# 2) Google Drive mounten
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# We will use timm for pretrained models (ConvNeXt, EfficientNet etc.)
!pip -q install timm
import timm

# Set deterministic seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Use GPU if available
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)


Using device: cuda


In [20]:
# Path to the metadata CSV and DICOM folder
VINBIGDATA_DIR = '/content/drive/MyDrive/Colab/IKIM_CXR/data/VinBigData'
meta_path = f'{VINBIGDATA_DIR}/train.csv'
train_img_dir = f'{VINBIGDATA_DIR}/train'

MODEL_DIR = "/content/drive/MyDrive/Colab/IKIM_CXR/models/vinbigdata_classifier"
OUTPUT_DIR = "/content/drive/MyDrive/Colab/IKIM_CXR/outputs/vinbigdata_classifier"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Modelle werden gespeichert in:", MODEL_DIR)
print("Ergebnisse werden gespeichert in:", OUTPUT_DIR)

# Load the annotations
annotations = pd.read_csv(meta_path)

# Add full DICOM paths.
# Your files are expected to be stored as:
# /content/drive/MyDrive/Colab/IKIM_CXR/data/VinBigData/train/<image_id>.dicom
annotations['image_path'] = annotations['image_id'].apply(
    lambda x: f'{train_img_dir}/{x}.dicom'
)

# Safety check: show missing image files, if any
missing_paths = annotations[['image_id', 'image_path']].drop_duplicates()
missing_paths = missing_paths[~missing_paths['image_path'].apply(os.path.exists)]

print(f"Annotations rows: {len(annotations)}")
print(f"Unique images in train.csv: {annotations['image_id'].nunique()}")
print(f"Missing DICOM files: {len(missing_paths)}")

if len(missing_paths) > 0:
    print("Example missing paths:")
    display(missing_paths.head())

# Use Kaggle's official class_id order instead of alphabetical class_name sorting
class_map = (
    annotations[['class_id', 'class_name']]
    .drop_duplicates()
    .sort_values('class_id')
    .reset_index(drop=True)
)

class_names = class_map['class_name'].tolist()
class_to_idx = dict(zip(class_map['class_name'], class_map['class_id']))
num_classes = len(class_names)

print('Number of classes:', num_classes)
display(class_map)

# Aggregate bounding-box rows to image-level multi-label targets
# Each image may have multiple annotations, possibly from multiple radiologists.
# We create one 15-dimensional binary vector per image.
label_df = (
    annotations
    .groupby('image_id')['class_id']
    .apply(lambda x: sorted(set(x.astype(int).tolist())))
    .reset_index()
)

def build_target_vector(class_id_list):
    target = np.zeros(num_classes, dtype=np.float32)
    for class_id in class_id_list:
        target[class_id] = 1.0
    return target

label_df['target'] = label_df['class_id'].apply(build_target_vector)

# Add one image_path per image
image_paths = annotations[['image_id', 'image_path']].drop_duplicates()
label_df = label_df.merge(image_paths, on='image_id', how='left')

print(label_df.head())
print("Final label_df shape:", label_df.shape)

Modelle werden gespeichert in: /content/drive/MyDrive/Colab/IKIM_CXR/models/vinbigdata_classifier
Ergebnisse werden gespeichert in: /content/drive/MyDrive/Colab/IKIM_CXR/outputs/vinbigdata_classifier
Annotations rows: 67914
Unique images in train.csv: 15000
Missing DICOM files: 0
Number of classes: 15


,class_id,class_name
0,0,Aortic enlargement
1,1,Atelectasis
2,2,Calcification
3,3,Cardiomegaly
4,4,Consolidation
5,5,ILD
6,6,Infiltration
7,7,Lung Opacity
8,8,Nodule/Mass
9,9,Other lesion


                           image_id           class_id  \
0  000434271f63a053c4128a0ba6352c7f               [14]   
1  00053190460d56c53cc3e57321387478               [14]   
2  0005e8e3701dfb1dd93d53e2ff537b6e       [4, 6, 7, 8]   
3  0006e0a85696f6bb578e84fafa9a5607               [14]   
4  0007d316f756b3fa0baea2ff514ce945  [0, 3, 5, 11, 13]   

                                              target  \
0  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
1  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
2  [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, ...   
3  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
4  [1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, ...   

                                          image_path  
0  /content/drive/MyDrive/Colab/IKIM_CXR/data/Vin...  
1  /content/drive/MyDrive/Colab/IKIM_CXR/data/Vin...  
2  /content/drive/MyDrive/Colab/IKIM_CXR/data/Vin...  
3  /content/drive/MyDrive/Colab/IKIM_CXR/data/Vin...  
4  /content/drive/MyDrive/Colab/IKIM_CX

In [21]:

# Compute class prevalence
positive_counts = np.zeros(num_classes)
total_images = len(label_df)

for targets in label_df['target']:
    positive_counts += targets

neg_counts = total_images - positive_counts

# pos_weight = (num_negatives / num_positives); clip to avoid huge values (max 20)
pos_weight = neg_counts / (positive_counts + 1e-6)
pos_weight = np.minimum(pos_weight, 20.0)

print('Positive class weights:', pos_weight)

# Convert to a tensor for BCEWithLogitsLoss
pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float32).to(DEVICE)


Positive class weights: [ 3.89077274 20.         20.          5.52173913 20.         20.
 20.         10.34644477 17.15980627 12.22751322 13.53488371  6.57193336
 20.          8.27643784  0.4142938 ]


In [22]:

# Function to perform iterative stratification for multi-label data
# We use iterative_train_test_split from the 'skmultilearn' package for multi-label stratification.
# If not installed, install it via pip.
!pip -q install scikit-multilearn
from skmultilearn.model_selection import iterative_train_test_split

# Prepare feature matrix (dummy indices) and label matrix for stratification
X = label_df[['image_id']].values
Y = np.stack(label_df['target'].values)

# First, split into train+val and test (20% test)
X_temp, Y_temp, X_test, Y_test = iterative_train_test_split(X, Y, test_size=0.2)

# Split temp into train and validation (approximately 10% val of the whole dataset)
val_size = 0.125  # because 0.125 * 0.8 ≈ 0.10
X_train, Y_train, X_val, Y_val = iterative_train_test_split(X_temp, Y_temp, test_size=val_size)

# Convert back to DataFrame for convenience
train_ids = [x[0] for x in X_train]
val_ids = [x[0] for x in X_val]
test_ids = [x[0] for x in X_test]

train_df = label_df[label_df['image_id'].isin(train_ids)].reset_index(drop=True)
val_df   = label_df[label_df['image_id'].isin(val_ids)].reset_index(drop=True)
test_df  = label_df[label_df['image_id'].isin(test_ids)].reset_index(drop=True)

print('Train size:', len(train_df), 'Val size:', len(val_df), 'Test size:', len(test_df))


Train size: 10490 Val size: 1500 Test size: 3010


In [23]:
%pip install -q pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg timm scikit-multilearn

import pydicom
import cv2

# Preprocessing transform common to train, val and test
def clahe_grayscale(img):
    # Apply slight Gaussian blur and CLAHE to improve contrast
    blur = cv2.GaussianBlur(img, (3, 3), 0)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img_clahe = clahe.apply(blur)
    return img_clahe

# Albumentations transforms for training and validation
train_transform = A.Compose([
    A.RandomResizedCrop(
        size=(384, 384),
        scale=(0.9, 1.0),
        ratio=(0.9, 1.1),
        p=1.0
    ),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=10, p=0.5),
    A.OneOf([
        A.RandomBrightnessContrast(
            brightness_limit=0.1,
            contrast_limit=0.1,
            p=0.5
        ),
        A.CLAHE(
            clip_limit=2.0,
            tile_grid_size=(8, 8),
            p=0.5
        ),
    ], p=0.5),
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(height=384, width=384),
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2(),
])

class VinBigDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Read DICOM image
        dcm_path = row["image_path"]
        dcm = pydicom.dcmread(dcm_path)

        img = dcm.pixel_array.astype(np.float32)

        # Apply rescale slope/intercept if present
        if "RescaleSlope" in dcm and "RescaleIntercept" in dcm:
            img = img * float(dcm.RescaleSlope) + float(dcm.RescaleIntercept)

        # Normalize to 0-255 and convert to uint8
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        img = (img * 255.0).astype(np.uint8)

        # Invert for MONOCHROME1
        if getattr(dcm, "PhotometricInterpretation", "") == "MONOCHROME1":
            img = 255 - img

        # Apply CLAHE preprocessing
        img = clahe_grayscale(img)

        # Apply transforms
        if self.transform:
            augmented = self.transform(image=img)
            img = augmented["image"]

        # Expand to three channels by repeating grayscale channel
        img = img.repeat(3, 1, 1)

        # Targets
        target = torch.tensor(row["target"], dtype=torch.float32)

        return img, target

# Create dataset instances
train_dataset = VinBigDataset(train_df, transform=train_transform)
val_dataset   = VinBigDataset(val_df, transform=val_transform)
test_dataset  = VinBigDataset(test_df, transform=val_transform)

# DataLoaders
batch_size = 8  # adjust for your GPU memory

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [24]:
# Define a classifier using a timm backbone
# For Colab, convnext_tiny is safer. convnext_base is heavier.
backbone_name = "convnext_tiny"

model = timm.create_model(
    backbone_name,
    pretrained=True,
    num_classes=num_classes,
    in_chans=3
)

model = model.to(DEVICE)

# Loss function – weighted BCE with per-class pos_weight
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

# Optimiser
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# Scheduler: one step per epoch, not per batch
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=10
)

print("Model:", backbone_name)
print("Number of output classes:", num_classes)

# Quick shape test before training
imgs, targets = next(iter(train_loader))
print("Input batch shape:", imgs.shape)
print("Target batch shape:", targets.shape)

with torch.no_grad():
    test_logits = model(imgs[:2].to(DEVICE))

print("Model output shape:", test_logits.shape)

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

Model: convnext_tiny
Number of output classes: 15
Input batch shape: torch.Size([8, 3, 384, 384])
Target batch shape: torch.Size([8, 15])
Model output shape: torch.Size([2, 15])


In [25]:

def mixup_data(x, y, alpha=0.4):
    '''Returns mixed inputs and targets for mixup.  y is expected to be shape [batch_size, num_classes].'''
    if alpha > 0.0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    mixed_y = lam * y + (1 - lam) * y[index]
    return mixed_x, mixed_y


In [ ]:

def train_one_epoch(model, loader, optimizer, criterion, scheduler=None, mixup_prob=0.3):
    model.train()
    running_loss = 0.0
    for imgs, targets in tqdm(loader, desc='Training', leave=False):
        imgs = imgs.to(DEVICE)
        targets = targets.to(DEVICE)
        # Apply mixup with given probability
        if random.random() < mixup_prob:
            imgs, targets = mixup_data(imgs, targets)

        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        if scheduler is not None:
            scheduler.step()
        running_loss += loss.item() * imgs.size(0)
    return running_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_targets = []
    all_probs = []
    for imgs, targets in tqdm(loader, desc='Evaluating', leave=False):
        imgs = imgs.to(DEVICE)
        targets = targets.to(DEVICE)
        logits = model(imgs)
        probs = torch.sigmoid(logits)
        all_targets.append(targets.cpu().numpy())
        all_probs.append(probs.cpu().numpy())
    all_targets = np.concatenate(all_targets, axis=0)
    all_probs = np.concatenate(all_probs, axis=0)

    # Compute per-class AUROC; handle classes with no positive samples
    per_class_auroc = []
    for i in range(num_classes):
        if (all_targets[:, i].sum() > 0) and ((1 - all_targets[:, i]).sum() > 0):
            auc = roc_auc_score(all_targets[:, i], all_probs[:, i])
        else:
            auc = np.nan
        per_class_auroc.append(auc)

    # Macro and micro AUROC
    valid_aucs = [x for x in per_class_auroc if not np.isnan(x)]
    macro_auroc = np.mean(valid_aucs)
    micro_auroc = roc_auc_score(all_targets.ravel(), all_probs.ravel())

    return macro_auroc, micro_auroc, per_class_auroc

# Main training loop
num_epochs = 10  # Adjust as needed

best_macro = 0.0
history = []

for epoch in range(num_epochs):
    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        scheduler
    )

    val_macro, val_micro, val_per_class = evaluate(model, val_loader)

    print(
        f"Epoch {epoch + 1}/{num_epochs} - "
        f"Train loss: {train_loss:.4f} - "
        f"Val macro AUROC: {val_macro:.4f} - "
        f"Val micro AUROC: {val_micro:.4f}"
    )

    # Trainingshistorie nach jeder Epoche sammeln
    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_macro_auc": val_macro,
        "val_micro_auc": val_micro
    })

    # Bestes Modell speichern
    if val_macro > best_macro:
        best_macro = val_macro

        torch.save(
            model.state_dict(),
            f"{MODEL_DIR}/best_model.pth"
        )

        print(f"Saved new best model to: {MODEL_DIR}/best_model.pth")

# Trainingshistorie am Ende als CSV speichern
history_df = pd.DataFrame(history)

history_df.to_csv(
    f"{OUTPUT_DIR}/training_history.csv",
    index=False
)

print(f"Training history saved to: {OUTPUT_DIR}/training_history.csv")
print(f"Best validation macro AUROC: {best_macro:.4f}")


Training:  49%|████▊     | 639/1312 [1:35:48<1:28:07,  7.86s/it]

In [ ]:
# Load the best checkpoint
model.load_state_dict(
    torch.load(f"{MODEL_DIR}/best_model.pth", map_location=DEVICE)
)
model.eval()

# Evaluate on the internal test split from train.csv
test_macro, test_micro, test_per_class = evaluate(model, test_loader)

print(f"Internal test macro AUROC: {test_macro:.4f}")
print(f"Internal test micro AUROC: {test_micro:.4f}")

# Save test metrics
test_metrics = pd.DataFrame({
    "metric": ["macro_auroc", "micro_auroc"],
    "value": [test_macro, test_micro]
})

test_metrics.to_csv(
    f"{OUTPUT_DIR}/test_metrics.csv",
    index=False
)

# Save per-class AUROC
per_class_auc_df = pd.DataFrame({
    "class_name": class_names,
    "auroc": test_per_class
})

per_class_auc_df.to_csv(
    f"{OUTPUT_DIR}/test_per_class_auc.csv",
    index=False
)

print(f"Test metrics saved to: {OUTPUT_DIR}/test_metrics.csv")
print(f"Per-class AUROC saved to: {OUTPUT_DIR}/test_per_class_auc.csv")

# Example: generate predictions for a few internal test images
sample_loader = DataLoader(test_dataset, batch_size=4, shuffle=True)

imgs, targets = next(iter(sample_loader))
imgs = imgs.to(DEVICE)

with torch.no_grad():
    logits = model(imgs)
    probs = torch.sigmoid(logits).cpu().numpy()

for i in range(len(imgs)):
    print("True labels:", [class_names[j] for j in range(num_classes) if targets[i][j] == 1])
    print("Top predicted labels:", [class_names[j] for j in np.argsort(-probs[i])[:5]])
    print("-" * 40)